# 01 – Base model: Age, Gender & Ethnicity (UTKFace)
This is the **base model** that every internship task builds on.

**Kaggle setup**
1. *Add Input* → search **utkface-new** (by jangedoo) → Add.
2. *Settings → Accelerator* → **GPU T4 x2** (or P100).
3. *Run All*. When it finishes, open the **Output** tab and download the `models/` folder.

Each file is named `age_gender_ethnicity_date.jpg`:
- gender: 0 = Male, 1 = Female
- ethnicity: 0 = White, 1 = Black, 2 = Asian, 3 = Indian, 4 = Others

## 1. Importing modules

In [ ]:
import os, json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Conv2D, Dropout, Flatten, MaxPooling2D,
                                     Input, BatchNormalization, RandomFlip)
from tensorflow.keras import initializers, callbacks
from tensorflow.keras.utils import plot_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Exploring the dataset

In [ ]:
# Kaggle sometimes mounts the dataset at a different path - find the folder automatically
candidates = [Path('/kaggle/input/utkface-new/UTKFace'), Path('/kaggle/input/utkface-new/utkface_aligned_cropped/UTKFace')]
candidates += list(Path('/kaggle/input').glob('**/UTKFace'))
BASE_DIR = next(p for p in candidates if p.exists())
print('Dataset folder:', BASE_DIR)

GENDER = {0: 'Male', 1: 'Female'}
ETHNICITY = {0: 'White', 1: 'Black', 2: 'Asian', 3: 'Indian', 4: 'Others'}

rows = []
for f in sorted(os.listdir(BASE_DIR)):
    parts = f.split('_')
    if len(parts) < 4:            # a few files are missing the ethnicity label - skip them
        continue
    try:
        age, gender, eth = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        continue
    if gender in GENDER and eth in ETHNICITY:
        rows.append((str(BASE_DIR / f), age, gender, eth))

df = pd.DataFrame(rows, columns=['image', 'age', 'gender', 'ethnicity'])
print(df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(df['age'], bins=50, kde=True, ax=ax[0]).set_title('Age distribution')
sns.countplot(x=df['gender'].map(GENDER), ax=ax[1]).set_title('Gender')
sns.countplot(x=df['ethnicity'].map(ETHNICITY), ax=ax[2]).set_title('Ethnicity')
plt.show()

print('People over 60:', (df.age > 60).sum(), '| aged 20-30:', df.age.between(20, 30).sum())

In [ ]:
plt.figure(figsize=(15, 15))
for i, (_, row) in enumerate(df.sample(25, random_state=1).iterrows()):
    plt.subplot(5, 5, i + 1)
    plt.imshow(load_img(row['image']))
    plt.title(f"{row['age']} | {GENDER[row['gender']]} | {ETHNICITY[row['ethnicity']]}", fontsize=10)
    plt.axis('off')
plt.show()

## 3. Feature extraction
Every face becomes a 128×128 grayscale array scaled to 0–1. This same preprocessing is used later by every task.

In [ ]:
IMG_SIZE = 128

def extract_features(images):
    feats = np.zeros((len(images), IMG_SIZE, IMG_SIZE, 1), dtype='float32')
    for i, p in enumerate(tqdm(images)):
        img = load_img(p, color_mode='grayscale').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        feats[i, :, :, 0] = np.asarray(img, dtype='float32') / 255.0
    return feats

X = extract_features(df['image'].tolist())
y_gender = df['gender'].values.astype('float32')
y_eth = df['ethnicity'].values
y_age = df['age'].values.astype('float32')
print(X.shape)

In [ ]:
# 80% train, 10% validation, 10% test (stratified by ethnicity so every group is in each split)
idx = np.arange(len(X))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y_eth)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED, stratify=y_eth[temp_idx])

def split(i):
    return X[i], {'gender_out': y_gender[i], 'ethnicity_out': y_eth[i], 'age_out': y_age[i]}

X_train, y_train = split(train_idx)
X_val, y_val = split(val_idx)
X_test, y_test = split(test_idx)
print(len(X_train), len(X_val), len(X_test))

## 4. Model creation
One shared CNN backbone, three output heads: gender (binary), ethnicity (5 classes), age (regression in years).

In [ ]:
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding='same', activation='relu',
               kernel_initializer=initializers.HeNormal())(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    return x

def head(x, units, name, activation):
    h = Dense(256, activation='relu')(x)
    h = Dropout(0.4)(h)
    return Dense(units, activation=activation, name=name)(h)

def build_model(img_size=128):
    inputs = Input(shape=(img_size, img_size, 1), name='face')
    x = RandomFlip('horizontal')(inputs)          # augmentation: only active while training
    for f in (32, 64, 128, 256, 256):
        x = conv_block(x, f)                       # 128 -> 4x4 after five blocks
    x = Flatten()(x)
    x = Dropout(0.3)(x)
    model = Model(inputs, [head(x, 1, 'gender_out', 'sigmoid'),
                           head(x, 5, 'ethnicity_out', 'softmax'),
                           head(x, 1, 'age_out', 'relu')], name='age_gender_ethnicity')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={'gender_out': 'binary_crossentropy',
              'ethnicity_out': 'sparse_categorical_crossentropy',
              'age_out': 'mae'},
        # age loss is in years (~5-10), the others are ~0.3-1, so scale age down to balance them
        loss_weights={'gender_out': 1.0, 'ethnicity_out': 1.0, 'age_out': 0.1},
        metrics={'gender_out': ['accuracy'], 'ethnicity_out': ['accuracy'], 'age_out': ['mae']})
    return model

model = build_model(IMG_SIZE)
model.summary()

In [ ]:
plot_model(model, show_shapes=True, show_layer_names=True, dpi=60)

## 5. Model training

In [ ]:
EPOCHS = 60
os.makedirs('models', exist_ok=True)
cbs = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    batch_size=64, epochs=EPOCHS, callbacks=cbs)

## 6. Visualization & evaluation

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, key, title in zip(ax, ['gender_out_accuracy', 'ethnicity_out_accuracy', 'age_out_mae'],
                         ['Gender accuracy', 'Ethnicity accuracy', 'Age MAE (years)']):
    a.plot(h[key], label='train'); a.plot(h['val_' + key], label='validation')
    a.set_title(title); a.set_xlabel('epoch'); a.legend()
plt.show()

In [ ]:
# Final scores on the test set (images the model never saw during training)
res = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
print(f"Gender accuracy:    {res['gender_out_accuracy']:.3f}")
print(f"Ethnicity accuracy: {res['ethnicity_out_accuracy']:.3f}")
print(f"Age MAE:            {res['age_out_mae']:.2f} years")

In [ ]:
pg, pe, pa = model.predict(X_test, verbose=0)
yt_eth = y_test['ethnicity_out']
print(classification_report(yt_eth, pe.argmax(1), target_names=list(ETHNICITY.values())))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(yt_eth, pe.argmax(1)), annot=True, fmt='d', cmap='Blues',
            xticklabels=ETHNICITY.values(), yticklabels=ETHNICITY.values(), ax=ax[0])
ax[0].set_title('Ethnicity confusion matrix'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')
ax[1].scatter(y_test['age_out'], pa.ravel(), s=4, alpha=0.4)
ax[1].plot([0, 100], [0, 100], 'r--'); ax[1].set_xlabel('True age'); ax[1].set_ylabel('Predicted age')
ax[1].set_title('Age: predicted vs true')
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
for i, k in enumerate(np.random.choice(len(X_test), 12, replace=False)):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[k].squeeze(), cmap='gray'); plt.axis('off')
    plt.title(f"True: {y_test['age_out'][k]:.0f}, {GENDER[int(y_test['gender_out'][k])]}, {ETHNICITY[yt_eth[k]]}\n"
              f"Pred: {pa[k][0]:.0f}, {GENDER[int(pg[k][0] > 0.5)]}, {ETHNICITY[pe[k].argmax()]}", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Save the model
Saved without the optimizer state (~16 MB) so it fits on GitHub. Download `models/` from the Output tab.

In [ ]:
# Keras 3 always stores the optimizer with a compiled model, so save an uncompiled copy (3x smaller)
export = Model(model.inputs, model.outputs, name=model.name)   # same layers and weights, no optimizer
export.save('models/base_age_gender_ethnicity.keras')
with open('models/labels.json', 'w') as f:
    json.dump({'img_size': IMG_SIZE, 'color_mode': 'grayscale',
               'gender': GENDER, 'ethnicity': ETHNICITY}, f, indent=2)
print('Saved:', {p: f'{os.path.getsize(os.path.join("models", p)) / 1e6:.1f} MB' for p in os.listdir('models')})

In [ ]:
# Check the saved file loads and gives the same predictions
reloaded = tf.keras.models.load_model('models/base_age_gender_ethnicity.keras', compile=False)
a = model.predict(X_test[:5], verbose=0); b = reloaded.predict(X_test[:5], verbose=0)
print('Reload OK:', all(np.allclose(x, y, atol=1e-4) for x, y in zip(a, b)))